In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Tue Mar 31 19:47:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.18             Driver Version: 580.126.18     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:04:00.0 Off |                  N/A |
|  0%   45C    P8             15W /  350W |       0MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# For Qwen2.5-VL
# # %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     %pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# %pip install transformers==4.56.2
# %pip install --no-deps trl==0.22.2

# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 83.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 30.7 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 21.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 42.8 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 46.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 51.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 55.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 45.9 MB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 53.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.4 MB

In [4]:
# # For Kaggle setup
# %pip install unsloth
# %pip install --upgrade "torchvision>=0.25.0"

# Configuration

In [5]:
SEED = 42

# Model configuration
# MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'
MODEL_ID = 'unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit'

# Visual codebook configuration
K = 8192

BVIR_DATA_ID = 'alxxtexxr/BVIR-Data'
# CODEBOOK_PATH = 'model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'
CODEBOOK_PATH = 'model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy'

In [6]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print(f"Random seed: {seed}")

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Random seed: 42


In [7]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [8]:
import copy

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    # load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    dtype = torch.float16, # Force FP16
    # device_map = 'cpu', # Load in CPU first to not explode the memory usage in GPU when expanding the token embeddings
)
tokenizer_orig = copy.deepcopy(tokenizer) # Copy the original tokenizer for comparison later

print(model)

==((====))==  Unsloth 2026.3.18: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.559 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [9]:
# Ensure the embeddings are in float16, not bfloat16, 
# to make it more universal for different types of GPUs
print(model.model.language_model.embed_tokens.weight.dtype)
print(model.lm_head.weight.dtype)
print(model.device)

torch.float16
torch.float16
cuda:0


In [10]:
# Calculate LM embedding norms to set codebook scaling
E = model.get_input_embeddings().weight.data.clone()
lm_norms = E.norm(dim=1)
print("Embedding norms (mean, std):", lm_norms.mean().item(), lm_norms.std().item())

# Randomly sample K norms with replacement
target_norms = lm_norms[torch.randint(0, lm_norms.size(0), (K,))]
print("Target norms (mean, std):", target_norms.mean().item(), target_norms.std().item())

Embedding norms (mean, std): 1.322265625 0.30029296875
Target norms (mean, std): 1.3203125 0.30224609375


In [11]:
def check_embs_info():
    print("Input embeddings shape:", model.get_input_embeddings().weight.shape)
    print("Output embeddings shape:", model.get_output_embeddings().weight.shape)
    print("Weight-tied:", model.get_input_embeddings().weight is model.get_output_embeddings().weight)

check_embs_info()

Input embeddings shape: torch.Size([151936, 4096])
Output embeddings shape: torch.Size([151936, 4096])
Weight-tied: False


In [12]:
list(tokenizer.tokenizer.get_added_vocab().items())[:30]

[('<|endoftext|>', 151643),
 ('<|im_start|>', 151644),
 ('<|im_end|>', 151645),
 ('<|object_ref_start|>', 151646),
 ('<|object_ref_end|>', 151647),
 ('<|box_start|>', 151648),
 ('<|box_end|>', 151649),
 ('<|quad_start|>', 151650),
 ('<|quad_end|>', 151651),
 ('<|vision_start|>', 151652),
 ('<|vision_end|>', 151653),
 ('<|vision_pad|>', 151654),
 ('<|image_pad|>', 151655),
 ('<|video_pad|>', 151656),
 ('<tool_call>', 151657),
 ('</tool_call>', 151658),
 ('<|fim_prefix|>', 151659),
 ('<|fim_middle|>', 151660),
 ('<|fim_suffix|>', 151661),
 ('<|fim_pad|>', 151662),
 ('<|repo_name|>', 151663),
 ('<|file_sep|>', 151664),
 ('<tool_response>', 151665),
 ('</tool_response>', 151666),
 ('<think>', 151667),
 ('</think>', 151668)]

In [13]:
def check_emb_stats(i):
    inner_model = getattr(model, 'model', model)
    x = inner_model.language_model.embed_tokens.weight[i].cpu().detach().numpy()
    print("i:", i)
    print(f"mean: {x.mean()}, max: {x.max()}, min: {x.min()}")
    print("First values:", x[:5])
    print()

check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding

i: 0
mean: -0.0001690387725830078, max: 0.1484375, min: -0.1591796875
First values: [ 0.04565  0.02832  0.02429 -0.01318 -0.01941]

i: 151664
mean: 4.684925079345703e-05, max: 0.035888671875, min: -0.03662109375
First values: [-0.00775   0.00479  -0.002426  0.007385 -0.005707]



In [14]:
vtoks = [f'<|vtok_{i}|>' for i in range(K)]
tokenizer.tokenizer.add_tokens(vtoks);

In [15]:
vtok_start_id, vtok_end_id = tokenizer.tokenizer.encode(f'<|vtok_0|><|vtok_{K-1}|>')
print("Visual token start ID:", vtok_start_id)
print("Visual token end ID:", vtok_end_id)

Visual token start ID: 151669
Visual token end ID: 159860


In [16]:
model.resize_token_embeddings(vtok_end_id+1, mean_resizing=False)
print(model)

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [17]:
# model = model.to("cuda")
# torch.cuda.empty_cache()
# # TODO: Clear the CPU memory too

In [18]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: -0.0001690387725830078, max: 0.1484375, min: -0.1591796875
First values: [ 0.04565  0.02832  0.02429 -0.01318 -0.01941]

i: 151664
mean: 4.684925079345703e-05, max: 0.035888671875, min: -0.03662109375
First values: [-0.00775   0.00479  -0.002426  0.007385 -0.005707]

i: 151669
mean: -9.077787399291992e-05, max: 0.0277099609375, min: -0.0277099609375
First values: [-0.000534 -0.001915 -0.0027   -0.001579 -0.003113]

i: 159860
mean: -3.635883331298828e-05, max: 0.07379150390625, min: -0.0738525390625
First values: [-0.02147   0.005352 -0.034     0.04623  -0.01245 ]



In [19]:
from huggingface_hub import hf_hub_download

codebook_path = hf_hub_download(
    repo_id=BVIR_DATA_ID,
    filename=CODEBOOK_PATH,
    repo_type='dataset',
)
print("Visual codebook downloaded to:", codebook_path)

model_unsloth_Qwen3-VL-8B-Instruct-unslo(…):   0%|          | 0.00/134M [00:00<?, ?B/s]

Visual codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--BVIR-Data/snapshots/eb30c21d6b504493ffedb974b8ad932f0192efe9/model_unsloth_Qwen3-VL-8B-Instruct-unsloth-bnb-4bit_fp16.data_sft_jxie_coco_captions_1000.data_rl_alxxtexxr_ViRL1.25K_1000/codebook_faiss_8192.npy


In [23]:
import numpy as np

codebook = torch.from_numpy(np.load(codebook_path)).to(torch.float16).to(model.device) # (K, 3584) -> Current: (8192, 3584)
assert codebook.shape[0] == K, f"Expected codebook with {K} entries, but got {codebook.shape[0]}"

print(f"Visual codebook shape: {codebook.shape}, dtype: {codebook.dtype}, device: {codebook.device}")
print("Visual codebook norms before scaling (mean, std):", codebook.norm(dim=1).mean().item(), codebook.norm(dim=1).std().item())

Visual codebook shape: torch.Size([8192, 4096]), dtype: torch.float16, device: cuda:0
Visual codebook norms before scaling (mean, std): 0.81396484375 0.06103515625


In [24]:
# Normalize codebook to unit vectors first
codebook_norm = codebook / codebook.norm(dim=1, keepdim=True) 
print("Visual codebook norms after normalization (mean, std):", codebook_norm.norm(dim=1).mean().item(), codebook_norm.norm(dim=1).std().item())

# Scale to LM-like norms
codebook_scaled = codebook_norm * target_norms[:, None]              
print("Visual codebook norms after scaling (mean, std):", codebook_scaled.norm(dim=1).mean().item(), codebook_scaled.norm(dim=1).std().item())

Visual codebook norms after normalization (mean, std): 1.0 0.00014269351959228516
Visual codebook norms after scaling (mean, std): 1.3203125 0.30224609375


In [25]:
chunk_size = 512

with torch.no_grad():
    for i in range(0, vtok_end_id - vtok_start_id + 1, chunk_size):
        a_start = vtok_start_id + i
        a_end = min(a_start + chunk_size, vtok_end_id + 1)

        b_start = i
        b_end = b_start + (a_end - a_start)
        
        print(f"Processing chunk: codebook_scaled[{b_start}:{b_end}] => embed_tokens[{a_start}:{a_end}]")

        # Move only a chunk to GPU
        chunk = codebook_scaled[b_start:b_end].to(
            model.lm_head.weight.device,
            dtype=model.lm_head.weight.dtype,
            non_blocking=True
        )

        # Replace in the input embeddings
        getattr(model, 'model', model).language_model.embed_tokens.weight[a_start:a_end] = chunk
        
        # Replace in the output embeddings too even if they are not tied, to give better initialization
        model.lm_head.weight[a_start:a_end] = chunk

        # Optional: free reference quickly
        del chunk

Processing chunk: codebook_scaled[0:512] => embed_tokens[151669:152181]
Processing chunk: codebook_scaled[512:1024] => embed_tokens[152181:152693]
Processing chunk: codebook_scaled[1024:1536] => embed_tokens[152693:153205]
Processing chunk: codebook_scaled[1536:2048] => embed_tokens[153205:153717]
Processing chunk: codebook_scaled[2048:2560] => embed_tokens[153717:154229]
Processing chunk: codebook_scaled[2560:3072] => embed_tokens[154229:154741]
Processing chunk: codebook_scaled[3072:3584] => embed_tokens[154741:155253]
Processing chunk: codebook_scaled[3584:4096] => embed_tokens[155253:155765]
Processing chunk: codebook_scaled[4096:4608] => embed_tokens[155765:156277]
Processing chunk: codebook_scaled[4608:5120] => embed_tokens[156277:156789]
Processing chunk: codebook_scaled[5120:5632] => embed_tokens[156789:157301]
Processing chunk: codebook_scaled[5632:6144] => embed_tokens[157301:157813]
Processing chunk: codebook_scaled[6144:6656] => embed_tokens[157813:158325]
Processing chunk:

In [26]:
model.get_input_embeddings().weight is model.get_output_embeddings().weight

False

In [27]:
check_emb_stats(0) # Check the stats of the first token embedding
check_emb_stats(151664) # Check the stats of the last added token embedding
check_emb_stats(152063) # Check the stats of the last embedding in the vocabulary
check_emb_stats(vtok_start_id) # Check the stats of the first virtual token embedding
check_emb_stats(vtok_end_id) # Check the stats of the last virtual token embedding

i: 0
mean: -0.0001690387725830078, max: 0.1484375, min: -0.1591796875
First values: [ 0.04565  0.02832  0.02429 -0.01318 -0.01941]

i: 151664
mean: 4.684925079345703e-05, max: 0.035888671875, min: -0.03662109375
First values: [-0.00775   0.00479  -0.002426  0.007385 -0.005707]

i: 152063
mean: 2.580881118774414e-05, max: 0.07464599609375, min: -0.12200927734375
First values: [-0.01051    0.0001126 -0.03293    0.00849    0.002857 ]

i: 151669
mean: -1.3649463653564453e-05, max: 0.104248046875, min: -0.1092529296875
First values: [0.01962  0.02322  0.01195  0.01997  0.009384]

i: 159860
mean: -4.00543212890625e-05, max: 0.20947265625, min: -0.1937255859375
First values: [ 0.0116  -0.01746 -0.05374 -0.01198 -0.01444]



In [30]:
# Verify that the added embeddings match the scaled codebook
embed_tokens_added = getattr(model, 'model', model).language_model.embed_tokens.weight[-K:]
lm_head_added = model.lm_head.weight[-K:]

assert torch.allclose(codebook_scaled, embed_tokens_added) and torch.allclose(codebook_scaled, lm_head_added), "The added embeddings do not match the scaled codebook!"

def check_stats(x):
    print(f"mean: {x.mean().item()}, max: {x.max().item()}, min: {x.min().item()}")

print("Last scaled codebook embedding stats:")
check_stats(codebook_scaled[-1])
print()
print("Last input embedding stats:")
check_stats(embed_tokens_added[-1])
print()
print("Last output embedding stats:")
check_stats(lm_head_added[-1])

Last scaled codebook embedding stats:
mean: -4.00543212890625e-05, max: 0.20947265625, min: -0.1937255859375

Last input embedding stats:
mean: -4.00543212890625e-05, max: 0.20947265625, min: -0.1937255859375

Last output embedding stats:
mean: -4.00543212890625e-05, max: 0.20947265625, min: -0.1937255859375


In [31]:
model.config.vocab_size = len(tokenizer.tokenizer)
print("Updated model config vocab size:", model.config.vocab_size)

Updated model config vocab size: 159861


In [38]:
# HUB_MODEL_ID = f"alxxtexxr/{MODEL_ID.split('/')[-1]}-with-codebook"
HUB_MODEL_ID = f"alxxtexxr/{MODEL_ID.split('/')[-1].replace('-unsloth-bnb-4bit', '')}-with-codebook"

model.push_to_hub(HUB_MODEL_ID, max_shard_size='6GB')
tokenizer.push_to_hub(HUB_MODEL_ID)

README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/alxxtexxr/Qwen3-VL-8B-Instruct-with-codebook


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            